# 第１弾 STEP 1: Iceberg Table への書き込みと S3 物理構造の確認

## 目的
- Snowflake から Iceberg Table を作成・書き込みする
- S3 上の物理ファイル（metadata / manifest / data）の構造を確認する
- 書き込みのたびに metadata が更新されるスナップショットの仕組みを理解する

## 使用データ
Snowflake サンプルデータ（`SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS`）の一部を使用する

## STEP 0: 接続確認

In [ ]:
%%sql -r connection_check
-- 接続・権限確認
SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE();

In [ ]:
%%sql -r ext_vol_check
-- External Volume の確認
SHOW EXTERNAL VOLUMES LIKE 'ICEBERG_S3_VOLUME';

In [ ]:
%%sql -r ext_vol_desc
-- External Volume の詳細確認（S3 の storage_location が正しいか）
DESC EXTERNAL VOLUME ICEBERG_S3_VOLUME;

In [ ]:
%%sql -r dataframe_1
-- リセット: Iceberg テーブルを削除
-- ※ S3 上のファイル削除は Snowflake が非同期で行うため、DROP 直後は残っている場合がある
-- ※ 完全にクリーンにしたい場合は AWS 側で BASE_LOCATION ディレクトリを手動削除すること
-- ※ CREATE OR REPLACE は新しい BASE_LOCATION サブディレクトリを生成するため、旧ファイルが残っていても動作に影響なし
ALTER ICEBERG TABLE IF EXISTS ICEBERG_DB.WORK.ORDERS_ICEBERG SET DATA_RETENTION_TIME_IN_DAYS = 0;
DROP ICEBERG TABLE IF EXISTS ICEBERG_DB.WORK.ORDERS_ICEBERG;

## STEP 1: 書き込み用 DB / Schema に切り替え

In [ ]:
%%sql -r use_context
USE DATABASE ICEBERG_DB;
USE SCHEMA WORK;
USE WAREHOUSE SANDBOX_WH;
CREATE FILE FORMAT IF NOT EXISTS ICEBERG_DB.WORK.JSON_FF TYPE = JSON;

## STEP 2: サンプルデータの確認

In [ ]:
%%sql -r tpch_count
-- TPCH サンプルデータの確認（件数・スキーマ）
SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS;

In [ ]:
%%sql -r tpch_sample
-- サンプル表示
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
WHERE O_ORDERKEY <= 40000
ORDER BY O_ORDERKEY
LIMIT 10;

## STEP 3: Iceberg Table の作成

Snowflake managed Iceberg Table を作成する。  
`EXTERNAL_VOLUME` に作成済みの `ICEBERG_S3_VOLUME` を指定する。

In [ ]:
%%sql -r create_iceberg
CREATE OR REPLACE ICEBERG TABLE ICEBERG_DB.WORK.ORDERS_ICEBERG (
    O_ORDERKEY      NUMBER(38, 0),
    O_CUSTKEY       NUMBER(38, 0),
    O_ORDERSTATUS   STRING,
    O_TOTALPRICE    NUMBER(12, 2),
    O_ORDERDATE     DATE,
    O_ORDERPRIORITY STRING,
    O_CLERK         STRING,
    O_SHIPPRIORITY  NUMBER(38, 0),
    O_COMMENT       STRING
)
EXTERNAL_VOLUME = 'ICEBERG_S3_VOLUME'
CATALOG = 'SNOWFLAKE'
BASE_LOCATION = 'orders_iceberg/';

In [ ]:
%%sql -r show_iceberg
-- テーブルが作成されたか確認
SHOW ICEBERG TABLES IN SCHEMA ICEBERG_DB.WORK;

In [ ]:
import json
row = session.sql("SHOW ICEBERG TABLES LIKE 'ORDERS_ICEBERG' IN SCHEMA ICEBERG_DB.WORK").collect()
base_location = row[0]['base_location']
stage_path = f'@ICEBERG_DB.WORK.ICEBERG_WORK_STAGE/{base_location}'
print(f'base_location = {base_location}')
print(f'stage_path    = {stage_path}')

In [ ]:
%%sql -r s3_files_after_create
-- S3 ファイル一覧（CREATE TABLE 直後）
LIST {{stage_path}};

In [ ]:
%%sql -r metadata_after_create
-- 最新 metadata.json の中身（CREATE TABLE 直後）
SELECT $1 AS metadata_json
FROM {{stage_path}}metadata/ (FILE_FORMAT => 'ICEBERG_DB.WORK.JSON_FF', PATTERN => '.*metadata\\.json')
ORDER BY METADATA$FILENAME DESC
LIMIT 1;

## STEP 4: データの書き込み（1回目）

In [ ]:
%%sql -r insert_1996
-- 1回目: 1996年のデータを挿入（ID ベースでサンプリング）
INSERT INTO ICEBERG_DB.WORK.ORDERS_ICEBERG
SELECT
    O_ORDERKEY, O_CUSTKEY, O_ORDERSTATUS, O_TOTALPRICE,
    O_ORDERDATE, O_ORDERPRIORITY, O_CLERK, O_SHIPPRIORITY, O_COMMENT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
WHERE YEAR(O_ORDERDATE) = 1996
  AND O_ORDERKEY <= 40000;

In [ ]:
%%sql -r count_after_1996
-- 件数確認
SELECT COUNT(*) FROM ICEBERG_DB.WORK.ORDERS_ICEBERG;

In [ ]:
%%sql -r s3_files_after_insert1
-- S3 ファイル一覧（1回目 INSERT 後）
LIST {{stage_path}};

In [ ]:
%%sql -r metadata_after_insert1
-- 最新 metadata.json の中身（1回目 INSERT 後）
SELECT $1 AS metadata_json
FROM {{stage_path}}metadata/ (FILE_FORMAT => 'ICEBERG_DB.WORK.JSON_FF', PATTERN => '.*metadata\\.json')
ORDER BY METADATA$FILENAME DESC
LIMIT 1;

## STEP 6: データの書き込み（2回目）とスナップショットの変化

In [ ]:
%%sql -r insert_1997
-- 2回目: 1997年のデータを追加（ID ベースでサンプリング）
INSERT INTO ICEBERG_DB.WORK.ORDERS_ICEBERG
SELECT
    O_ORDERKEY, O_CUSTKEY, O_ORDERSTATUS, O_TOTALPRICE,
    O_ORDERDATE, O_ORDERPRIORITY, O_CLERK, O_SHIPPRIORITY, O_COMMENT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
WHERE YEAR(O_ORDERDATE) = 1997
  AND O_ORDERKEY <= 40000;

In [ ]:
%%sql -r count_total
-- 合計件数確認（約3,000件になるはず）
SELECT COUNT(*) FROM ICEBERG_DB.WORK.ORDERS_ICEBERG;

In [ ]:
%%sql -r s3_files_after_insert2
-- S3 ファイル一覧（2回目 INSERT 後）
LIST {{stage_path}};

In [ ]:
%%sql -r metadata_after_insert2
-- 最新 metadata.json の中身（2回目 INSERT 後）
SELECT $1 AS metadata_json
FROM {{stage_path}}metadata/ (FILE_FORMAT => 'ICEBERG_DB.WORK.JSON_FF', PATTERN => '.*metadata\\.json')
ORDER BY METADATA$FILENAME DESC
LIMIT 1;

## STEP 7: UPDATE / DELETE の挙動確認

Iceberg は更新・削除でも新しいスナップショットが作られる（COW: Copy-On-Write）。

In [ ]:
%%sql -r update_result
-- UPDATE: 特定ステータスの注文を更新
UPDATE ICEBERG_DB.WORK.ORDERS_ICEBERG
SET O_ORDERSTATUS = 'X'
WHERE YEAR(O_ORDERDATE) = 1996
  AND O_ORDERKEY < 30000;

In [ ]:
%%sql -r s3_files_after_update
-- S3 ファイル一覧（UPDATE 後）
LIST {{stage_path}};

In [ ]:
%%sql -r metadata_after_update
-- 最新 metadata.json の中身（UPDATE 後）
SELECT $1 AS metadata_json
FROM {{stage_path}}metadata/ (FILE_FORMAT => 'ICEBERG_DB.WORK.JSON_FF', PATTERN => '.*metadata\\.json')
ORDER BY METADATA$FILENAME DESC
LIMIT 1;

## まとめ・気づき

- INSERT / UPDATE / DELETE のたびに新しいスナップショットが S3 の metadata/ 以下に追加される
- 実データは data/ 以下の Parquet ファイルとして格納される
- Snowflake Managed Iceberg は `CATALOG = 'SNOWFLAKE'` で Snowflake が catalog を管理する
- タイムトラベルや読み取り実験は `02_read_and_timetravel.ipynb` へ